# Luna: real-trajectory component diagnostics

Inspect **saved** runs of the CLLP policy, the exact-board generator, their composition, and value along verified offline Sokoban trajectories. This notebook makes no model requests. The target and reference continuation come from the offline trajectory for evaluation only; the future actions are not supplied to the model.

Run `scripts.evaluate_trajectory_components` to produce new JSON reports. Default paths below refer to the documented live diagnostics. The default reports use the same five exploratory test layouts; prepare a new disjoint holdout for final claims.

## 1. Load the reports

Select the project kernel. Change these paths after running your own diagnostics. Missing reports are skipped so you can inspect one component independently. `status` and errors matter: partial runs must not be mistaken for completed evaluation.

In [ ]:
import sys
from pathlib import Path

# Usually automatic; set this only if the kernel starts outside the checkout/workspace.
REPO_PATH: str | None = None  # e.g. '/home/mtyrolski/phd/2026/CaRL'
anchors = (Path.cwd(), *Path.cwd().parents, *Path(sys.executable).parents)
candidates = (
    [Path(REPO_PATH).expanduser()]
    if REPO_PATH is not None
    else [candidate for anchor in anchors for candidate in (anchor, anchor / 'CaRL')]
)
ROOT = next((p.resolve() for p in candidates if (p / 'carl/icl').is_dir()), None)
if ROOT is None:
    raise RuntimeError(
        f'Cannot find the CaRL checkout from {Path.cwd()}. '
        'Set REPO_PATH above to the directory containing carl/icl, '
        'and select its .venv/bin/python as the notebook kernel.'
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'CaRL checkout: {ROOT} | Python: {sys.executable}')

# The checkout must be on sys.path before importing its modules.
# pylint: disable=wrong-import-position
import json  # noqa: E402 -- checkout path is configured above.

import pandas as pd  # noqa: E402 -- checkout path is configured above.
import plotly.graph_objects as go  # noqa: E402 -- checkout path is configured above.
from IPython.display import display  # noqa: E402 -- checkout path is configured above.

from carl.icl.boards import from_rows  # noqa: E402 -- checkout path is configured above.
from carl.icl.evaluation.board_graphics import board_comparison  # noqa: E402 -- checkout path is configured above.

# pylint: enable=wrong-import-position

POLICY_REPORT = ROOT / 'analysis/luna_icl/trajectory_tuning_20261102/final_simple/policy.json'
GENERATOR_REPORT = ROOT / 'analysis/luna_icl/trajectory_tuning_20261102/final_simple/generator.json'
GENERATED_POLICY_REPORT = ROOT / 'analysis/luna_icl/trajectory_tuning_20261102/final_simple/generated_policy.json'
VALUE_REPORT = ROOT / 'analysis/luna_icl/trajectory_tuning_20261102/final_simple/value.json'
paths = {'policy': POLICY_REPORT, 'generator': GENERATOR_REPORT,
         'generated_policy': GENERATED_POLICY_REPORT, 'value': VALUE_REPORT}
reports = {name: json.loads(path.read_text(encoding='utf-8'))
           for name, path in paths.items() if path.exists()}
for name, path in paths.items():
    print(f'{name}: {path}' + ('' if path.exists() else ' (not present)'))
summary = []
for name, report in reports.items():
    summary.append({'component': name, 'provider': report['provider'], 'status': report['status'],
                    **report['summary'], **{f"usage_{key}": value for key, value in report['metrics'].items()}})
display(pd.DataFrame(summary))

## 2. CLLP: exact hits on offline subgoals

Each target is an actual future board `k` actions along the reference trajectory. The sampled policy runs for at most `k` simulator actions and succeeds only on an exact board hit. Task completion is recorded separately. One seed per case is a diagnostic, not a reliable estimate of the full hit probability.

In [ ]:
if 'policy' in reports:
    rows = reports['policy']['cases']
    horizons = sorted({row['horizon'] for row in rows})
    totals = [sum(row['horizon'] == k for row in rows) for k in horizons]
    hits = [sum(row['horizon'] == k and row.get('reached', False) for row in rows) for k in horizons]
    fig = go.Figure(go.Bar(x=[str(k) for k in horizons], y=[h/n if n else 0 for h, n in zip(hits, totals)],
                           text=[f'{h}/{n}' for h, n in zip(hits, totals)], textposition='outside'))
    fig.update_layout(template='plotly_white', title='Exact subgoal hits per trajectory offset',
                      xaxis_title='Reference offset k', yaxis={'title': 'Observed fraction', 'range': [0, 1.15]})
    fig.show()
    display(pd.DataFrame([{key: row.get(key) for key in ('instance', 'position', 'horizon', 'reached', 'solved', 'actions_attempted', 'error')}
                          for row in rows]))

### Inspect a policy rollout

Choose a recorded case and a step. The first figure compares the input with the exact offline target; the second shows the actual simulator transition chosen by the model.

In [ ]:
POLICY_CASE = 0
POLICY_STEP = 0
if 'policy' in reports:
    cases = reports['policy']['cases']
    if not 0 <= POLICY_CASE < len(cases):
        raise ValueError('POLICY_CASE is outside the report')
    case = cases[POLICY_CASE]
    board_comparison(from_rows(case['state']), from_rows(case['target']),
                     ('Current board', 'Exact offline subgoal')).show()
    steps = case.get('steps', [])
    if steps:
        if not 0 <= POLICY_STEP < len(steps):
            raise ValueError('POLICY_STEP is outside this rollout')
        before = case['state'] if POLICY_STEP == 0 else steps[POLICY_STEP - 1]['state']
        step = steps[POLICY_STEP]
        board_comparison(from_rows(before), from_rows(step['state']),
                         (f'Before step {POLICY_STEP + 1}', f"After {step['action']}")).show()
        display(pd.DataFrame(steps).drop(columns=['state']))
    else:
        print('No action was executed:', case.get('error'))

## 3. Generator: exact-board outputs and actual policy rollout

The generator returns a full board. Basic structural validation rejects malformed or unchanged boards and can request a corrected response. There is no simulator-based reachability search or hidden proposal repair. The separate `generated_policy` report shows whether the actual CLLP rollout reached each generated target in at most `k` environment actions; a missing target remains a failure.

In [ ]:
if 'generator' in reports:
    rows = reports['generator']['cases']
    categories = ('valid proposal', 'no valid proposal', 'error')
    counts = [sum(row.get('valid_count', 0) > 0 for row in rows),
              sum(not row.get('valid_count', 0) and not row.get('error') for row in rows),
              sum(bool(row.get('error')) for row in rows)]
    fig = go.Figure(go.Bar(x=list(categories), y=counts, text=counts, textposition='outside'))
    fig.update_layout(template='plotly_white', title='Generated full boards', yaxis_title='Cases')
    fig.show()
    display(pd.DataFrame([{key: row.get(key) for key in ('instance', 'horizon', 'valid_count', 'matches_reference', 'diagnostics', 'error')}
                          for row in rows]))
if 'generated_policy' in reports:
    print('End-to-end generated-target policy:', reports['generated_policy']['summary'])
    display(pd.DataFrame([{key: row.get(key) for key in ('instance', 'horizon', 'reached', 'actions_attempted', 'reason', 'error')}
                          for row in reports['generated_policy']['cases']]))

### Inspect one generated board

Change `GENERATOR_CASE` and rerun. This shows the complete returned board, not an embedding or latent representation.

In [ ]:
GENERATOR_CASE = 0
if 'generator' in reports:
    cases = reports['generator']['cases']
    if not 0 <= GENERATOR_CASE < len(cases):
        raise ValueError('GENERATOR_CASE is outside the report')
    row = cases[GENERATOR_CASE]
    if 'proposal' in row:
        board_comparison(from_rows(row['state']), from_rows(row['proposal']),
                         ('Current board', 'Generated board')).show()
    else:
        print('No valid proposal for this case:', row.get('diagnostics'), row.get('error'))

## 4. Value along the reference trajectories

The dashed line is `γ ** remaining_actions` for the recorded solution witness; it is **not** necessarily the optimal value. The model estimates remaining actions from the board and training demonstrations, and the code applies `γ ** n`. Terminal values are also produced by the model; no simulator override is applied. Value need not increase on every real move: some solutions move a box away from a goal temporarily.

In [ ]:
if 'value' in reports:
    rows = reports['value']['cases']
    fig = go.Figure()
    for identifier in sorted({row['instance'] for row in rows}):
        group = sorted((row for row in rows if row['instance'] == identifier and 'value' in row), key=lambda x: x['position'])
        fig.add_trace(go.Scatter(x=[row['position'] for row in group], y=[row['value'] for row in group],
                                 mode='lines+markers', name=f'{identifier} estimated', legendgroup=identifier))
        fig.add_trace(go.Scatter(x=[row['position'] for row in group], y=[row['reference_return'] for row in group],
                                 mode='lines', line={'dash': 'dash'}, name=f'{identifier} witness', legendgroup=identifier))
    fig.update_layout(template='plotly_white', title='Value against trajectory progress', xaxis_title='Trajectory step',
                      yaxis={'title': 'Discounted continuation estimate', 'range': [0, 1.05]})
    fig.show()
    display(pd.DataFrame([{key: row.get(key) for key in ('instance', 'position', 'remaining', 'value', 'reference_return', 'terminal', 'error')}
                          for row in rows]))
    print('Spearman correlations by trajectory:', reports['value']['summary'].get('spearman_by_trajectory'))

## 5. Costs and interpretation

Logical requests include cache hits and retries. `usage_requests` counts actual provider attempts; token usage is reported by the provider. Only the actual CLLP actions are executed to test whether generated targets are reached; no separate reachability search is run. Reusing a cache changes physical cost. Before a final research claim, prepare a new disjoint holdout: these diagnostics have been inspected during development.